In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from vncorenlp import VnCoreNLP
import pandas as pd
import re
import os

llm = ChatOllama(model="myaniu/qwen2.5-1m")  # hoặc "llama3:8b-instruct"

def summary_abstractive(text, grade, length):
    prompt = [
        SystemMessage(content=f"Bạn là giáo viên dạy môn học tiếng Việt cho học sinh tiểu học lớp {grade} tại Việt Nam."),
        HumanMessage(content=f"""
        Đầu tiên tôi muốn bạn biết tóm tắt diễn giải là gì ?. Tóm tắt diễn giải (Abstractive Summarization): là phương pháp tóm tắt trong đó hệ thống hiểu toàn bộ nội dung tổng thể và tái diễn đạt lại các ý chính ngắn gọn và tự nhiên trong một văn bản mới, tương tự như cách con người tóm tắt. Hệ thống sẽ tạo ra các câu văn ngắn gọn, mạch lạc, tự nhiên và dễ hiểu.
        Tiếp theo tôi muốn bạn đọc đoạn văn sau và thực hiện tóm tắt diễn giải văn bản sao cho phù hợp với học sinh lớp {grade}.
        Với các yêu cầu như sau:
        - Tóm tắt diễn giải phải đảm bảo tính chính xác, tính logic, tính hợp lý, không mất ý nghĩa của văn bản gốc.
        - HÃY TUÂN THỦ VÀ THỰC HIỆN TÓM TẮT VĂN BẢN TRONG {length} CÂU. Phải chú trọng đến độ liên kết giữa các câu, giữa các đoạn văn. Nội dung mạch lạc, trôi chảy, liên kết. Bản tóm tắt phải dễ hiểu, lời lẽ tự nhiên, từ vựng phù hợp, đơn giản với học sinh lớp {grade}.
        - Cuối cùng hãy chuẩn hóa bản tóm tắt thành dạng văn bản hoàn chỉnh. Viết trên cùng 1 dòng.
        - KHÔNG THÊM CÁC DÁNH DẤU, CHỈ MỤC, CHÚ THÍCH, GHI CHÚ, ...
        - KẾT QUẢ CHỈ GỒM VĂN BẢN ĐƯỢC TÓM TẮT. KHÔNG THÊM TẤT CẢ CÁC YẾU TỐ, THÔNG TIN NGOÀI LỀ KHÁC.
        ĐOẠN VĂN:
        {text}

        KẾT QUẢ:
        """)
    ]
    response = llm(prompt)
    return response.content.strip()

vncorenlp = VnCoreNLP(
    "../../VnCoreNLP-master/VnCoreNLP-1.2.jar",
    annotators="wseg"
)

def tokenize_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    sentences = vncorenlp.tokenize(text)
    tokens = []
    for sent in sentences:
        tokens.extend(sent)
    return tokens

def load_grade_vocab(grade, vocab_dir="../../../datasets/grade_vocab"):
    vocab = set()
    for g in range(1, grade + 1):
        path = os.path.join(vocab_dir, f"grade_{g}.txt")
        with open(path, encoding="utf-8") as f:
            for line in f:
                vocab.add(line.strip().lower())
    return vocab

def vocab_coverage(text, vocab):
    tokens = tokenize_text(text)
    if len(tokens) == 0:
        return 0.0
    valid = sum(1 for t in tokens if t in vocab)
    return valid / len(tokens)


data = pd.read_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\Data_DG_with_summary5.xlsx")
df = data.copy()
max_retry = 3
threshold = 0.85 #giai thich pp nao
annotator = VnCoreNLP(
    "../../VnCoreNLP-master/VnCoreNLP-1.2.jar",
    annotators="wseg",
    max_heap_size='-Xmx2g'
)

for idx, row in df.iterrows():
    if pd.isna(row["summary"]) or row["summary"] == "":
        content = str(row["content"])
        grade = int(row["grade"])
        sentences = annotator.tokenize(content)
        num_sentences = len(sentences)
        length = num_sentences // 3
        vocab = load_grade_vocab(grade)
        dict_result = {}
        for i in range(max_retry):
            summary_result = summary_abstractive(content, grade, length)
            score = vocab_coverage(summary_result, vocab)
            dict_result[i] = {
                "summary": summary_result,
                "score": score
            }
            if score >= threshold:
                break
        best_summary = max(dict_result.values(), key=lambda x: x["score"])["summary"] if dict_result else ""
        df.loc[idx, "summary"] = best_summary

df.to_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\Data_DG_with_summary6.xlsx", index=False)

In [ ]:
import pandas as pd

# df phải có đủ 3 cột
assert {"content", "summary", "grade"}.issubset(df.columns)

df = df.dropna().reset_index(drop=True)

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "VietAI/vit5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(tokenizer.model_max_length)  # ~512 tokens

from datasets import Dataset

MAX_INPUT_LEN = 512
MAX_TARGET_LEN = 128

def preprocess(example):
    prompt = f"Tóm tắt văn bản cho học sinh lớp {example['grade']}: {example['content']}"

    inputs = tokenizer(
        prompt,
        truncation=True,
        padding="max_length",
        max_length=MAX_INPUT_LEN
    )

    targets = tokenizer(
        example["summary"],
        truncation=True,
        padding="max_length",
        max_length=MAX_TARGET_LEN
    )

    inputs["labels"] = targets["input_ids"]
    return inputs

dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

tokenized_ds = dataset.map(preprocess)

import evaluate
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
bertscore = evaluate.load("bertscore")

def compute_metrics(eval_pred):
    preds, labels = eval_pred

    # Xử lý label padding
    labels = [[l for l in label if l != -100] for label in labels]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # -------- ROUGE --------
    rouge_result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    # -------- BLEU --------
    bleu_result = bleu.compute(
        predictions=[p.split() for p in decoded_preds],
        references=[[l.split()] for l in decoded_labels]
    )

    # -------- BERTScore --------
    bert_result = bertscore.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        lang="vi",
        model_type="bert-base-multilingual-cased"
    )

    return {
        "rouge1": rouge_result["rouge1"],
        "rouge2": rouge_result["rouge2"],
        "rougeL": rouge_result["rougeL"],

        "bleu": bleu_result["bleu"],

        "bertscore_precision": sum(bert_result["precision"]) / len(bert_result["precision"]),
        "bertscore_recall": sum(bert_result["recall"]) / len(bert_result["recall"]),
        "bertscore_f1": sum(bert_result["f1"]) / len(bert_result["f1"]),
    }

import optuna
from transformers import Trainer, TrainingArguments
from transformers import EarlyStoppingCallback

def objective(trial):

    lr = trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True)
    batch_size = trial.suggest_categorical("batch_size", [4, 8])
    epochs = trial.suggest_int("num_train_epochs", 3, 6)
    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.1)

    args = TrainingArguments(
        output_dir=f"./optuna_trial_{trial.number}",

        evaluation_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="no",

        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=epochs,
        weight_decay=weight_decay,

        # 🔥 CHỐNG OVERFITTING
        label_smoothing_factor=0.1,
        max_grad_norm=1.0,
        fp16=True,

        report_to="none"
    )

    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized_ds["train"],
        eval_dataset=tokenized_ds["test"],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()
    metrics = trainer.evaluate()

    return metrics["eval_rougeL"]

import pandas as pd

def save_metrics(trainer, filename="metrics.csv"):
    logs = trainer.state.log_history
    df_metrics = pd.DataFrame(logs)
    df_metrics.to_csv(filename, index=False)
    return df_metrics



study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

best_params = study.best_params
print(best_params)

final_args = TrainingArguments(
    output_dir="./vit5_final",

    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    learning_rate=best_params["learning_rate"],
    per_device_train_batch_size=best_params["batch_size"],
    per_device_eval_batch_size=best_params["batch_size"],
    num_train_epochs=best_params["num_train_epochs"],
    weight_decay=best_params["weight_decay"],

    # 🔒 ANTI-OVERFITTING
    label_smoothing_factor=0.1,
    max_grad_norm=1.0,
    fp16=True,
    gradient_checkpointing=True,

    load_best_model_at_end=True,
    metric_for_best_model="eval_rougeL",
    greater_is_better=True,

    save_total_limit=2,
    report_to="none"
)

final_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

final_trainer = Trainer(
    model=final_model,
    args=final_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

final_trainer.train()

final_trainer.save_model("./vit5_grade_summary")
tokenizer.save_pretrained("./vit5_grade_summary")

metrics_df = save_metrics(final_trainer, "vit5_metrics.csv")
metrics_df.head()

import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

plt.plot(metrics_df["epoch"], metrics_df["eval_rouge1"], label="ROUGE-1")
plt.plot(metrics_df["epoch"], metrics_df["eval_rougeL"], label="ROUGE-L")
plt.plot(metrics_df["epoch"], metrics_df["bertscore_f1"], label="BERTScore-F1")

plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Evaluation Metrics over Epochs")
plt.legend()
plt.grid(True)
plt.show()

[WinError 3] The system cannot find the path specified: '/kaggle/working/data-dgtx'
e:\Project_NguyenMinhVu_2211110063\Source\ai\Summary\Dien Giai


'cp' is not recognized as an internal or external command,
operable program or batch file.
'ls' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
import torch

def summarize(text, grade):
    final_model.eval()

    if grade < 1:
        grade = 1
    if grade > 5:
        grade = 5

    prompt = f"Tóm tắt văn bản cho học sinh lớp {grade}: {text}"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(final_model.device)

    with torch.no_grad():
        output = final_model.generate(
            **inputs,
            max_new_tokens=grade * 20,
            num_beams=4,
            length_penalty=1.3,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

text = "Cây xanh giúp không khí trong lành và bảo vệ môi trường sống của con người."
summary = summarize(text, grade=3)

print(summary)


In [ ]:
def summarize_batch(texts, grades, batch_size=8):
    final_model.eval()
    results = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        batch_grades = grades[i:i+batch_size]

        prompts = [
            f"Tóm tắt văn bản cho học sinh lớp {g}: {t}"
            for t, g in zip(batch_texts, batch_grades)
        ]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(final_model.device)

        with torch.no_grad():
            outputs = final_model.generate(
                **inputs,
                max_new_tokens=100,
                num_beams=4,
                length_penalty=1.3,
                repetition_penalty=1.2,
                no_repeat_ngram_size=3,
                early_stopping=True
            )

        summaries = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        results.extend(summaries)

    return results

df["generated_summary"] = summarize_batch(
    df["content"].tolist(),
    df["grade"].tolist()
)
